In [1]:
from sentence_transformers import SentenceTransformer

/opt/homebrew/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [3]:
from transformers import AutoTokenizer, AutoModelForMaskedLM
from FlagEmbedding import BGEM3FlagModel

In [30]:
model = BGEM3FlagModel('BAAI/bge-m3',  use_fp16=True) # Setting use_fp16 to True speeds up computation with a slight performance degradation

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

In [66]:
sentence_eng = "Orangutans are large, gentle apes that are native to Indonesia and Malaysia"
sentence_es = "Los orangutanes son simios grandes y gentiles que son nativos de Indonesia y Malasia"
sentence_nl = "De Nederlandse politiek is verdeeld over de vraag of de regering de coronamaatregelen moet versoepelen"

In [56]:
output_eng = model.encode(sentence_eng, return_colbert_vecs=True, return_sparse=True, return_dense=True) 
output_es = model.encode(sentence_es, return_colbert_vecs=True, return_sparse=True, return_dense=True)
output_nl = model.encode(sentence_nl, return_colbert_vecs=True, return_sparse=True, return_dense=True)

In [57]:
type(output_eng)
output_eng.keys()

dict_keys(['dense_vecs', 'lexical_weights', 'colbert_vecs'])

In [70]:
output_eng["dense_vecs"].shape

(1024,)

In [71]:
output_es["dense_vecs"].shape

(1024,)

In [60]:
print(model.convert_id_to_token(output_eng['lexical_weights']))
print(model.convert_id_to_token(output_es['lexical_weights']))
print(model.convert_id_to_token(output_nl['lexical_weights']))

{'Orang': 0.2566, 'utan': 0.3337, 's': 0.1582, 'are': 0.1447, 'large': 0.1663, ',': 0.0736, 'gent': 0.1794, 'le': 0.1096, 'a': 0.1467, 'pes': 0.2057, 'that': 0.06027, 'na': 0.1385, 'tive': 0.1561, 'to': 0.1259, 'Indonesia': 0.217, 'and': 0.1059, 'Malaysia': 0.2136}
{'Los': 0.1, 'orang': 0.2532, 'utan': 0.325, 'es': 0.2211, 'son': 0.1233, 'sim': 0.2189, 'ios': 0.1958, 'grandes': 0.1611, 'y': 0.0774, 'gentil': 0.185, 'que': 0.03226, 'na': 0.1147, 'tivos': 0.1305, 'de': 0.03064, 'Indonesia': 0.184, 'Mala': 0.1366, 'sia': 0.1499}
{'De': 0.01721, 'Nederlandse': 0.1871, 'politiek': 0.2236, 'is': 0.003433, 'verde': 0.1678, 'eld': 0.1826, 'over': 0.0735, 'vraag': 0.1244, 'of': 0.01715, 'regering': 0.2056, 'corona': 0.1917, 'maat': 0.08636, 'regel': 0.1632, 'en': 0.0798, 'moet': 0.109, 'verso': 0.09375, 'ep': 0.1141, 'elen': 0.177}


In [61]:
print(output_eng['lexical_weights'])
print(output_es['lexical_weights'])
print(output_nl['lexical_weights'])

defaultdict(<class 'int'>, {'20229': 0.2566, '17399': 0.3337, '7': 0.1582, '621': 0.1447, '21334': 0.1663, '4': 0.0736, '21507': 0.1794, '133': 0.1096, '10': 0.1467, '13569': 0.2057, '450': 0.06027, '24': 0.1385, '4935': 0.1561, '47': 0.1259, '3799': 0.217, '136': 0.1059, '3605': 0.2136})
defaultdict(<class 'int'>, {'3731': 0.1, '1482': 0.2532, '17399': 0.325, '90': 0.2211, '775': 0.1233, '10777': 0.2189, '5790': 0.1958, '9255': 0.1611, '113': 0.0774, '73382': 0.185, '41': 0.03226, '24': 0.1147, '26465': 0.1305, '8': 0.03064, '3799': 0.184, '16522': 0.1366, '3478': 0.1499})
defaultdict(<class 'int'>, {'262': 0.01721, '43750': 0.1871, '158029': 0.2236, '83': 0.003433, '17258': 0.1678, '19388': 0.1826, '645': 0.0735, '22830': 0.1244, '111': 0.01715, '43229': 0.2056, '109728': 0.1917, '27697': 0.08636, '47144': 0.1632, '33': 0.0798, '3476': 0.109, '18892': 0.09375, '4517': 0.1141, '25704': 0.177})


In [62]:
lexical_scores = model.compute_lexical_matching_score(output_eng['lexical_weights'], output_es['lexical_weights'])
print(lexical_scores)

0.1642608642578125


In [65]:
dense_scores_eng_es = model.colbert_score(output_eng['colbert_vecs'], output_es['colbert_vecs'])
dense_scores_eng_nl = model.colbert_score(output_eng['colbert_vecs'], output_nl['colbert_vecs'])
dense_scores_es_nl = model.colbert_score(output_es['colbert_vecs'], output_nl['colbert_vecs'])

print(dense_scores_eng_es)
print(dense_scores_eng_nl)
print(dense_scores_es_nl)

tensor(0.8062)
tensor(0.2428)
tensor(0.2364)


In [63]:
from sentence_transformers import SentenceTransformer

model2 = SentenceTransformer("BAAI/bge-m3")